<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/13_GES_Aware_Genomic_RAG_Cell_7C6_V2_Exact_LLM_Generation_Execution_API_Compatible.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {ROOT}\n'
        'Confirm that Google Drive is mounted and the project folder name is unchanged.'
    )

print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Install and verify the exact frozen OpenAI SDK

In [2]:
from __future__ import annotations

import importlib.metadata as importlib_metadata
import subprocess
import sys

FROZEN_OPENAI_VERSION = '2.45.0'

def installed_version(distribution_name: str) -> str | None:
    try:
        return importlib_metadata.version(distribution_name)
    except importlib_metadata.PackageNotFoundError:
        return None

observed_openai_version = installed_version('openai')

if observed_openai_version != FROZEN_OPENAI_VERSION:
    print(
        f'Installing frozen OpenAI SDK: openai=={FROZEN_OPENAI_VERSION} '
        f'(observed: {observed_openai_version})'
    )
    subprocess.run(
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '--quiet',
            '--upgrade',
            f'openai=={FROZEN_OPENAI_VERSION}',
        ],
        check=True,
    )
    observed_openai_version = installed_version('openai')

if observed_openai_version != FROZEN_OPENAI_VERSION:
    raise RuntimeError(
        f'Frozen OpenAI SDK version not active. '
        f'Expected {FROZEN_OPENAI_VERSION}; observed {observed_openai_version}.'
    )

print(f'OpenAI SDK version: {observed_openai_version} — VERIFIED')

OpenAI SDK version: 2.45.0 — VERIFIED


## 2. Imports, frozen identities, exact hashes, and fail-closed output paths

In [3]:
from collections import OrderedDict
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from typing import Any
import copy
import csv
import hashlib
import inspect
import json
import os
import re
import shutil
import tempfile
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

from openai import (
    OpenAI,
    APIConnectionError,
    APITimeoutError,
    RateLimitError,
)

NOTEBOOK_NAME = '13_GES_Aware_Genomic_RAG_Cell_7C6_V2_Exact_LLM_Generation_Execution_API_Compatible.ipynb'
CELL_ID = '7C6'
STAGE = '7C'
PACKAGE_VERSION = 'v2'

EXPECTED_PROMPTS = 480
EXPECTED_QUESTIONS = 80
EXPECTED_ALIASES = 6
EXPECTED_RUN_IDS = [0, 1, 2]
EXPECTED_REQUESTS = 1_440

FROZEN_MODEL = 'gpt-4.1-mini-2025-04-14'
FROZEN_API = 'Responses API'
FROZEN_TEMPERATURE = 0.0
FROZEN_TOP_P = 1.0
FROZEN_MAX_OUTPUT_TOKENS = 1200
FROZEN_PRESENCE_PENALTY = 0.0
FROZEN_FREQUENCY_PENALTY = 0.0
FROZEN_TIMEOUT_SECONDS = 120
FROZEN_MAX_TRANSPORT_RETRIES = 3
FROZEN_RETRY_BACKOFF_SECONDS = [2, 5, 10]

EXPECTED_CELL_7C5_TERMINAL_DECISION = (
    'PASS_STAGE7C5_COMPLETE_CELL7C4_480_SCORE_BLIND_PROMPTS_AND_CELL7B4_FIXED_LLM_RUNTIME_'
    'CONFIG_REVERIFIED_1440_REQUEST_GENERATION_PLAN_FROZEN_CHECKSUM_PROTECTED_CELL7C6_'
    'EXACT_LLM_GENERATION_ONLY_AUTHORIZED_NO_SCORE_BEARING_AUDIT_CELL7A3_SCORES_'
    'ANSWER_KEYS_ADJUDICATION_OR_RAG_METRICS'
)

EXPECTED_CELL_7C5_AUTHORIZATION_DECISION = (
    'AUTHORIZE_STAGE7C_CELL7C6_EXACT_1440_FROZEN_LLM_GENERATION_REQUESTS_'
    '480_SCORE_BLIND_PROMPTS_X3_RUNS_FIXED_GPT41MINI_20250414_RESPONSES_API_'
    'TEMPERATURE0_TOPP1_MAXTOKENS1200_STRICT_JSON_SCHEMA_NO_TOOLS_WEB_FILESEARCH_'
    'CODEINTERPRETER_PROMPT_MODIFICATION_ANSWER_KEYS_ADJUDICATION_OR_RAG_METRICS'
)


EXPECTED_CELL_7C5R_AUTHORIZATION_DECISION = (
    'AUTHORIZE_STAGE7C_CELL7C6_V2_EXACT_1440_FROZEN_LLM_GENERATION_REQUESTS_'
    'WITH_API_COMPATIBLE_STRUCTURED_OUTPUT_SCHEMA_PROJECTION_REMOVING_ONLY_'
    'EVIDENCE_IDS_UNIQUEITEMS_AND_PRESERVING_POSTHOC_UNIQUENESS_VALIDATION_'
    'NO_PROMPT_MODEL_GENERATION_SETTING_SCORE_ANSWER_KEY_OR_EVALUATION_CHANGE'
)

EXPECTED_CELL_7C5R_TERMINAL_DECISION = (
    'PASS_STAGE7C5R_OPENAI_STRUCTURED_OUTPUT_SCHEMA_COMPATIBILITY_REMEDIATION_'
    'ORIGINAL_CELL7B4_SCHEMA_REVERIFIED_ONLY_EVIDENCE_IDS_UNIQUEITEMS_REMOVED_'
    'FOR_API_SERIALIZATION_POSTHOC_UNIQUENESS_VALIDATION_PRESERVED_CHECKSUM_PROTECTED_'
    'CELL7C6_V2_EXACT_1440_GENERATION_ONLY_AUTHORIZED_NO_OTHER_SCIENTIFIC_CHANGE'
)

EXPECTED_SCIENTIFIC_SCHEMA_CANONICAL_SHA256 = (
    'a32bbf83a5f954d64c60ee4d737299f86be80692aab773c35ebd6f681f3aa533'
)
EXPECTED_API_SCHEMA_CANONICAL_SHA256 = (
    '527ab2c8b4e76974555b4c8cba99188e00b36a83334c4e190203f46b06238ef0'
)

# --------------------------------------------------------------------------------------------------
# Exact successful Cell 7C5 package.
# --------------------------------------------------------------------------------------------------
CELL_7C5_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c5_llm_generation_authorization_v1'
)
CELL_7C5_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c5_llm_generation_authorization_v1'
)

CELL_7C5 = OrderedDict([
    ('authorization', {
        'path': CELL_7C5_CONFIG_DIR / 'cell_7c5_stage7c_cell7c6_llm_generation_authorization_v1.json',
        'sha256': '8cb177929450933802e324ede4f00ffdb392c8d9f172b8692c28be28e3b81116',
    }),
    ('generation_plan', {
        'path': CELL_7C5_CONFIG_DIR / 'cell_7c5_frozen_generation_request_plan_v1.parquet',
        'sha256': '6fe2a7c8dcd2601cfe10e269827a77f9cc78d0fd9f83dfad0644a0efd74b41cf',
    }),
    ('input_inventory', {
        'path': CELL_7C5_CONFIG_DIR / 'cell_7c5_authorized_generation_input_inventory_v1.csv',
        'sha256': '8118dec995f09762c25f7ded6be9d77685a2b33d81fe05d7c2f6bd7e380f655d',
    }),
    ('qc', {
        'path': CELL_7C5_QC_DIR / 'cell_7c5_llm_generation_authorization_qc_v1.json',
        'sha256': '8f1eff5d5d6d39df5a32b709b235653a5a1dc26102850e875e673f72eef3900d',
    }),
    ('manifest', {
        'path': CELL_7C5_CONFIG_DIR / 'cell_7c5_llm_generation_authorization_manifest_v1.json',
        'sha256': '3bfb3a24fff0eb1ace3ff83d994c4f9ea4f0d0492ae014739378311fb48a51df',
    }),
])

# --------------------------------------------------------------------------------------------------
# Exact successful Cell 7C5R remediation package.
# --------------------------------------------------------------------------------------------------
CELL_7C5R_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c5r_structured_output_api_compatibility_remediation_v1'
)
CELL_7C5R_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c5r_structured_output_api_compatibility_remediation_v1'
)

CELL_7C5R = OrderedDict([
    ('api_compatible_schema', {
        'path': CELL_7C5R_CONFIG_DIR / 'cell_7c5r_api_compatible_response_schema_v1.json',
        'sha256': 'd56f0f14e20cd3071ba06e9fc8fd4efb8adee814143647108427546d0379fb09',
    }),
    ('remediation_authorization', {
        'path': CELL_7C5R_CONFIG_DIR / 'cell_7c5r_stage7c_cell7c6_v2_generation_remediation_authorization_v1.json',
        'sha256': '4606754e3e83bfd804738d6af8974108156d1cbc413131131d6491915f184e22',
    }),
    ('input_inventory', {
        'path': CELL_7C5R_CONFIG_DIR / 'cell_7c5r_remediation_input_inventory_v1.csv',
        'sha256': '23ff6cf66ca4a2f4f8da7c638223e65821f7acb7659c2731abf89f0d5f7ab9d5',
    }),
    ('qc', {
        'path': CELL_7C5R_QC_DIR / 'cell_7c5r_structured_output_api_compatibility_qc_v1.json',
        'sha256': 'e111b507ca5456e576c6e2024ffb5f233eb6c6c2319d76e2971292684af8fb60',
    }),
    ('manifest', {
        'path': CELL_7C5R_CONFIG_DIR / 'cell_7c5r_structured_output_api_compatibility_manifest_v1.json',
        'sha256': '580ef084f0e5470b83645fdc6334ff8f385fb93ae42502e98dbac180f2259b63',
    }),
])

# --------------------------------------------------------------------------------------------------
# Exact Cell 7C4 score-blind prompts.
# --------------------------------------------------------------------------------------------------
CELL_7C4_EXEC_DIR = (
    ROOT / 'outputs' / 'rag_execution' / 'stage7_rag'
    / 'cell_7c4_score_blind_prompt_materialization_v1'
)
CELL_7C4_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c4_score_blind_prompt_materialization_v1'
)

CELL_7C4_PROMPTS = {
    'path': CELL_7C4_EXEC_DIR / 'cell_7c4_score_blind_prompt_instances_v1.parquet',
    'sha256': 'd5b088815d7710fce5c79b89b39828787a745d360404d20270319e91fdcd4ce5',
}
CELL_7C4_MANIFEST = {
    'path': CELL_7C4_CONFIG_DIR / 'cell_7c4_prompt_materialization_manifest_v1.json',
    'sha256': 'a58c4b61e2c603a496047d5b36174d466e59cf293491115b364c6466480ba31d',
}

# --------------------------------------------------------------------------------------------------
# Exact Cell 7B4 frozen LLM configuration.
# --------------------------------------------------------------------------------------------------
CELL_7B4_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7b4_configuration_freeze_v1'
)
CELL_7B4_LLM = {
    'path': CELL_7B4_CONFIG_DIR / 'cell_7b4_llm_prompt_response_configuration_v1.json',
    'sha256': 'e3f684f9c471b8074dc41f2398f8aff0cb03217f188e2eab2ad8f4c0070a810b',
}
CELL_7B4_RUNTIME = {
    'path': CELL_7B4_CONFIG_DIR / 'cell_7b4_runtime_and_determinism_configuration_v1.json',
    'sha256': '6003c85ef151ae1d7dca530462fa1dbcf6983be4b7e92744b8e65c8d8b42b1d3',
}

# --------------------------------------------------------------------------------------------------
# Recovery-only checkpoint directory. It is intentionally excluded from the frozen final manifest.
# --------------------------------------------------------------------------------------------------
CHECKPOINT_DIR = (
    ROOT / 'outputs' / 'execution_checkpoints' / 'stage7_rag'
    / 'cell_7c6_exact_llm_generation_v2'
)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------------------------------
# Frozen final package.
# --------------------------------------------------------------------------------------------------
EXEC_DIR = (
    ROOT / 'outputs' / 'rag_execution' / 'stage7_rag'
    / 'cell_7c6_exact_llm_generation_v2'
)
QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c6_exact_llm_generation_v2'
)
CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c6_exact_llm_generation_v2'
)

OUTPUTS = OrderedDict([
    ('raw_response_envelopes',
     EXEC_DIR / 'cell_7c6_v2_raw_response_envelopes_v1.jsonl'),
    ('response_inventory',
     EXEC_DIR / 'cell_7c6_v2_generation_response_inventory_v1.parquet'),
    ('structured_outputs',
     EXEC_DIR / 'cell_7c6_v2_structured_response_outputs_v1.parquet'),
    ('input_inventory',
     CONFIG_DIR / 'cell_7c6_v2_verified_generation_input_inventory_v1.csv'),
    ('execution_report',
     QC_DIR / 'cell_7c6_v2_llm_generation_execution_report_v1.json'),
    ('qc',
     QC_DIR / 'cell_7c6_v2_llm_generation_qc_v1.json'),
    ('manifest',
     CONFIG_DIR / 'cell_7c6_v2_exact_llm_generation_manifest_v1.json'),
])

for directory in (EXEC_DIR, QC_DIR, CONFIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if OUTPUTS['manifest'].exists():
    raise FileExistsError(
        f'Frozen Cell 7C6 manifest already exists: {OUTPUTS["manifest"]}\\n'
        'Fail-closed overwrite protection is active.'
    )

unexpected_existing = [
    path for key, path in OUTPUTS.items()
    if key != 'manifest' and path.exists()
]
if unexpected_existing:
    raise FileExistsError(
        'One or more final Cell 7C6 artifacts already exist without a manifest. '
        'Fail closed rather than overwriting a potentially partial freeze:\\n'
        + '\\n'.join(str(path) for path in unexpected_existing)
    )

print(f'Checkpoint directory : {CHECKPOINT_DIR}')
print(f'Execution directory  : {EXEC_DIR}')
print(f'QC directory         : {QC_DIR}')
print(f'Config directory     : {CONFIG_DIR}')

Checkpoint directory : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/execution_checkpoints/stage7_rag/cell_7c6_exact_llm_generation_v2
Execution directory  : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/rag_execution/stage7_rag/cell_7c6_exact_llm_generation_v2
QC directory         : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7c6_exact_llm_generation_v2
Config directory     : /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7c6_exact_llm_generation_v2


## 3. Strict checksum, serialization, atomic-write, and checkpoint helpers

In [4]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def canonical_json_text(payload: Any) -> str:
    return json.dumps(
        to_json_native(payload),
        sort_keys=True,
        separators=(',', ':'),
        ensure_ascii=False,
        allow_nan=False,
    )


def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Empty SHA-256 sidecar: {path}')
    token = text.split()[0].strip()
    if not re.fullmatch(r'[0-9a-fA-F]{64}', token):
        raise ValueError(
            f'Invalid SHA-256 sidecar format: {path}\\n'
            f'Observed first token: {token!r}'
        )
    return token.lower()


def sidecar_is_valid(path: Path) -> bool:
    sc = sidecar_path(path)
    return (
        path.exists()
        and sc.exists()
        and read_sidecar_hash(sc) == sha256_file(path)
    )


def verify_exact_artifact(label: str, path: Path, expected_sha256: str) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f'Missing frozen artifact [{label}]: {path}')
    observed = sha256_file(path)
    if observed != expected_sha256:
        raise AssertionError(
            f'SHA-256 mismatch for {label}.\\n'
            f'Expected: {expected_sha256}\\n'
            f'Observed: {observed}\\n'
            f'Path: {path}'
        )
    if not sidecar_is_valid(path):
        raise AssertionError(f'Invalid/missing SHA-256 sidecar for {label}: {path}')
    return {
        'input_id': label,
        'path': str(path),
        'sha256': observed,
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)),
        'sidecar_valid': True,
    }


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def parquet_metadata(path: Path) -> dict[str, Any]:
    pf = pq.ParquetFile(path)
    return {
        'rows': int(pf.metadata.num_rows),
        'columns': int(pf.metadata.num_columns),
        'schema_names': list(pf.schema_arrow.names),
    }


def to_json_native(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, np.generic):
        return to_json_native(value.item())
    if isinstance(value, np.ndarray):
        return [to_json_native(v) for v in value.tolist()]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): to_json_native(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_json_native(v) for v in value]
    if hasattr(value, 'model_dump'):
        return to_json_native(value.model_dump())
    if hasattr(value, 'item'):
        return to_json_native(value.item())
    raise TypeError(f'Unsupported JSON type: {type(value).__name__}')


def atomic_write_bytes(path: Path, payload: bytes) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f'.tmp.{os.getpid()}')
    if tmp.exists():
        tmp.unlink()
    with tmp.open('wb') as handle:
        handle.write(payload)
        handle.flush()
        os.fsync(handle.fileno())
    tmp.replace(path)


def write_sidecar(path: Path) -> None:
    digest = sha256_file(path)
    payload = f'{digest}  {path.name}{chr(10)}'.encode('utf-8')
    atomic_write_bytes(sidecar_path(path), payload)


def stable_write_json(path: Path, payload: Any) -> str:
    text = json.dumps(
        to_json_native(payload),
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
    ) + chr(10)
    atomic_write_bytes(path, text.encode('utf-8'))
    return sha256_file(path)


def stable_write_jsonl(path: Path, records: list[dict[str, Any]]) -> str:
    lines = [
        canonical_json_text(record)
        for record in records
    ]
    payload = (chr(10).join(lines) + chr(10)).encode('utf-8')
    atomic_write_bytes(path, payload)
    return sha256_file(path)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f'.tmp.{os.getpid()}')
    frame.to_csv(
        tmp,
        index=False,
        encoding='utf-8',
        lineterminator=chr(10),
    )
    tmp.replace(path)
    return sha256_file(path)


def stable_write_parquet(path: Path, frame: pd.DataFrame) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f'.tmp.{os.getpid()}')
    frame.to_parquet(
        tmp,
        index=False,
        engine='pyarrow',
        compression='zstd',
    )
    tmp.replace(path)
    return sha256_file(path)


def checkpoint_path_for_request(generation_request_id: str) -> Path:
    safe_name = sha256_text(generation_request_id)
    return CHECKPOINT_DIR / f'{safe_name}.json'


def request_fingerprint(row: pd.Series) -> str:
    identity = {
        'cell_id': CELL_ID,
        'generation_request_id': str(row['generation_request_id']),
        'prompt_instance_id': str(row['prompt_instance_id']),
        'question_id': str(row['question_id']),
        'blinded_alias': str(row['blinded_alias']),
        'run_id': int(row['run_id']),
        'system_prompt_sha256': str(row['system_prompt_sha256']),
        'user_prompt_sha256': str(row['user_prompt_sha256']),
        'full_prompt_sha256': str(row['full_prompt_sha256']),
        'model_snapshot': str(row['model_snapshot']),
        'api': str(row['api']),
        'temperature': float(row['temperature']),
        'top_p': float(row['top_p']),
        'max_output_tokens': int(row['max_output_tokens']),
        'presence_penalty': float(row['presence_penalty']),
        'frequency_penalty': float(row['frequency_penalty']),
        'strict_json_schema': bool(row['strict_json_schema']),
        'response_schema_name': str(row['response_schema_name']),
        'response_schema_sha256': str(row['response_schema_sha256']),
    }
    return sha256_text(canonical_json_text(identity))


def idempotency_key_for_request(row: pd.Series) -> str:
    payload = (
        f'{CELL_ID}|{row["question_id"]}|{row["blinded_alias"]}|'
        f'{int(row["run_id"])}|{row["full_prompt_sha256"]}'
    )
    return sha256_text(payload)


# Regression-test the real-LF writer/sidecar behavior before any API call.
with tempfile.TemporaryDirectory(prefix='cell_7c6_writer_selftest_') as tmpdir_text:
    tmpdir = Path(tmpdir_text)
    test_json = tmpdir / 'test.json'
    stable_write_json(test_json, {'ok': True, 'value': 1})
    write_sidecar(test_json)
    if not test_json.read_bytes().endswith(bytes([10])):
        raise AssertionError('JSON writer did not emit a real LF byte.')
    if not sidecar_is_valid(test_json):
        raise AssertionError('SHA-256 sidecar self-test failed.')

print('Serialization / sidecar / checkpoint helpers: PASS')

Serialization / sidecar / checkpoint helpers: PASS


## 4. Reverify Cell 7C5 authorization, Cell 7C4 prompts, and Cell 7B4 LLM freeze

In [5]:
verified_inputs = []

for artifact_id, spec in CELL_7C5.items():
    record = verify_exact_artifact(
        f'cell_7c5_{artifact_id}',
        spec['path'],
        spec['sha256'],
    )
    record['source_cell'] = '7C5'
    verified_inputs.append(record)


for artifact_id, spec in CELL_7C5R.items():
    record = verify_exact_artifact(
        f'cell_7c5r_{artifact_id}',
        spec['path'],
        spec['sha256'],
    )
    record['source_cell'] = '7C5R'
    verified_inputs.append(record)

for artifact_id, spec in [
    ('cell_7c4_prompt_instances', CELL_7C4_PROMPTS),
    ('cell_7c4_manifest', CELL_7C4_MANIFEST),
    ('cell_7b4_llm_prompt_response', CELL_7B4_LLM),
    ('cell_7b4_runtime_determinism', CELL_7B4_RUNTIME),
]:
    record = verify_exact_artifact(
        artifact_id,
        spec['path'],
        spec['sha256'],
    )
    record['source_cell'] = artifact_id.split('_')[1].upper()
    verified_inputs.append(record)

authorization_7c5 = load_json(CELL_7C5['authorization']['path'])
qc_7c5 = load_json(CELL_7C5['qc']['path'])
manifest_7c5 = load_json(CELL_7C5['manifest']['path'])

authorization_7c5r = load_json(CELL_7C5R['remediation_authorization']['path'])
qc_7c5r = load_json(CELL_7C5R['qc']['path'])
manifest_7c5r = load_json(CELL_7C5R['manifest']['path'])
api_response_schema = load_json(CELL_7C5R['api_compatible_schema']['path'])

if manifest_7c5.get('terminal_decision') != EXPECTED_CELL_7C5_TERMINAL_DECISION:
    raise AssertionError('Cell 7C5 terminal decision mismatch.')
if manifest_7c5.get('next_authorized_cell') != '7C6':
    raise AssertionError('Cell 7C5 does not authorize Cell 7C6.')
if authorization_7c5.get('authorization_decision') != EXPECTED_CELL_7C5_AUTHORIZATION_DECISION:
    raise AssertionError('Cell 7C5 authorization decision mismatch.')
if authorization_7c5.get('authorized_cell', {}).get('cell_id') != '7C6':
    raise AssertionError('Cell 7C5 authorization target is not Cell 7C6.')
if authorization_7c5.get('answer_key_access_authorized') is not False:
    raise AssertionError('Cell 7C5 unexpectedly authorizes answer-key access.')
if authorization_7c5.get('evaluation_authorized') is not False:
    raise AssertionError('Cell 7C5 unexpectedly authorizes evaluation.')
if int(qc_7c5.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7C5 QC does not report zero failures.')

if manifest_7c5r.get('terminal_decision') != EXPECTED_CELL_7C5R_TERMINAL_DECISION:
    raise AssertionError('Cell 7C5R terminal decision mismatch.')
if manifest_7c5r.get('next_authorized_cell') != '7C6_V2':
    raise AssertionError('Cell 7C5R does not authorize Cell 7C6 V2.')
if authorization_7c5r.get('authorization_decision') != EXPECTED_CELL_7C5R_AUTHORIZATION_DECISION:
    raise AssertionError('Cell 7C5R remediation authorization decision mismatch.')
if authorization_7c5r.get('accepted_generation_requests_before_failure', None) is not None:
    pass
if int(qc_7c5r.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7C5R QC does not report zero failures.')
if authorization_7c5r.get('answer_key_access_authorized') is not False:
    raise AssertionError('Cell 7C5R unexpectedly authorizes answer-key access.')
if authorization_7c5r.get('evaluation_authorized') is not False:
    raise AssertionError('Cell 7C5R unexpectedly authorizes evaluation.')

llm_config = load_json(CELL_7B4_LLM['path'])
runtime_config = load_json(CELL_7B4_RUNTIME['path'])
llm_cfg = llm_config['llm']
generation_cfg = llm_config['generation']
structured_cfg = llm_config['structured_output']
scientific_response_schema = structured_cfg['schema']

def canonical_schema_sha256(schema: dict[str, Any]) -> str:
    return sha256_text(
        json.dumps(
            schema,
            sort_keys=True,
            separators=(',', ':'),
            ensure_ascii=False,
            allow_nan=False,
        )
    )

scientific_schema_sha256 = canonical_schema_sha256(scientific_response_schema)
api_schema_sha256 = canonical_schema_sha256(api_response_schema)

if scientific_schema_sha256 != EXPECTED_SCIENTIFIC_SCHEMA_CANONICAL_SHA256:
    raise AssertionError(
        'Original Cell 7B4 scientific response-schema canonical SHA-256 mismatch.'
    )
if api_schema_sha256 != EXPECTED_API_SCHEMA_CANONICAL_SHA256:
    raise AssertionError(
        'Cell 7C5R API-compatible response-schema canonical SHA-256 mismatch.'
    )

if (
    scientific_response_schema
    .get('properties', {})
    .get('evidence_ids', {})
    .get('uniqueItems')
) is not True:
    raise AssertionError(
        'Original scientific response schema no longer contains evidence_ids.uniqueItems=True.'
    )

if 'uniqueItems' in (
    api_response_schema
    .get('properties', {})
    .get('evidence_ids', {})
):
    raise AssertionError(
        'API-compatible response schema unexpectedly still contains evidence_ids.uniqueItems.'
    )

reconstructed_scientific_schema = copy.deepcopy(api_response_schema)
reconstructed_scientific_schema['properties']['evidence_ids']['uniqueItems'] = True

if reconstructed_scientific_schema != scientific_response_schema:
    raise AssertionError(
        'Cell 7C5R API-compatible schema differs from the original scientific schema '
        'by more than properties.evidence_ids.uniqueItems.'
    )

frozen_config_checks = OrderedDict([
    ('model_snapshot_exact', llm_cfg.get('model') == FROZEN_MODEL),
    ('responses_api_exact', llm_cfg.get('api') == FROZEN_API),
    ('fixed_snapshot_required', llm_cfg.get('fixed_snapshot_required') is True),
    ('tools_empty', llm_cfg.get('tools') == []),
    ('web_search_false', llm_cfg.get('web_search') is False),
    ('file_search_false', llm_cfg.get('file_search') is False),
    ('code_interpreter_false', llm_cfg.get('code_interpreter') is False),
    ('store_false', llm_cfg.get('store') is False),
    ('stream_false', llm_cfg.get('stream') is False),
    ('temperature_zero', generation_cfg.get('temperature') == FROZEN_TEMPERATURE),
    ('top_p_one', generation_cfg.get('top_p') == FROZEN_TOP_P),
    ('max_output_tokens_1200',
     generation_cfg.get('max_output_tokens') == FROZEN_MAX_OUTPUT_TOKENS),
    ('presence_penalty_zero',
     generation_cfg.get('presence_penalty') == FROZEN_PRESENCE_PENALTY),
    ('frequency_penalty_zero',
     generation_cfg.get('frequency_penalty') == FROZEN_FREQUENCY_PENALTY),
    ('three_repetitions',
     generation_cfg.get('repetitions_per_question_condition') == 3),
    ('run_ids_exact', generation_cfg.get('run_ids') == EXPECTED_RUN_IDS),
    ('timeout_exact',
     generation_cfg.get('timeout_seconds') == FROZEN_TIMEOUT_SECONDS),
    ('transport_retries_exact',
     generation_cfg.get('maximum_transport_retries') == FROZEN_MAX_TRANSPORT_RETRIES),
    ('retry_backoff_exact',
     generation_cfg.get('retry_backoff_seconds') == FROZEN_RETRY_BACKOFF_SECONDS),
    ('retry_policy_exact',
     generation_cfg.get('retry_policy') ==
     'retry transport/rate-limit failures only; never retry to obtain a preferred answer'),
    ('idempotency_rule_exact',
     generation_cfg.get('request_idempotency_key') ==
     'SHA256(cell_id|question_id|blinded_alias|run_id|prompt_sha256)'),
    ('strict_json_schema', structured_cfg.get('strict') is True),
    ('json_schema_type', structured_cfg.get('type') == 'json_schema'),
    ('structured_name_exact',
     structured_cfg.get('name') == 'ges_rag_genomic_evidence_answer'),
    ('additional_properties_false',
     scientific_response_schema.get('additionalProperties') is False),
])

failed_config_checks = [
    name for name, passed in frozen_config_checks.items()
    if not bool(passed)
]
if failed_config_checks:
    raise RuntimeError(
        'Frozen Cell 7B4 LLM configuration changed:\\n- '
        + '\\n- '.join(failed_config_checks)
    )

EXPECTED_RESPONSE_FIELDS = {
    'answer',
    'clinical_significance',
    'conflict_detected',
    'evidence_strength',
    'response_policy',
    'confidence',
    'evidence_ids',
    'reasoning_summary',
}
if set(scientific_response_schema.get('required', [])) != EXPECTED_RESPONSE_FIELDS:
    raise AssertionError('Frozen response required-field set changed.')
if set(scientific_response_schema.get('properties', {}).keys()) != EXPECTED_RESPONSE_FIELDS:
    raise AssertionError('Frozen response property set changed.')

print('Cell 7C5 package                         : 5/5 exact hashes + sidecars')
print('Cell 7C5R remediation package            : 5/5 exact hashes + sidecars')
print('Cell 7C5R terminal PASS                  : VERIFIED')
print('Cell 7C6 V2 remediation authorization    : VERIFIED')
print(f'Original scientific schema SHA-256       : {scientific_schema_sha256}')
print(f'API-compatible schema SHA-256            : {api_schema_sha256}')
print('Cell 7C5 terminal PASS                  : VERIFIED')
print('Cell 7C6 authorization                  : VERIFIED')
print('Cell 7C4 score-blind prompts            : VERIFIED')
print('Cell 7B4 exact LLM/runtime freeze       : VERIFIED')
print('Answer-key access                       : NOT AUTHORIZED')
print('RAG evaluation                          : NOT AUTHORIZED')

Cell 7C5 package                         : 5/5 exact hashes + sidecars
Cell 7C5R remediation package            : 5/5 exact hashes + sidecars
Cell 7C5R terminal PASS                  : VERIFIED
Cell 7C6 V2 remediation authorization    : VERIFIED
Original scientific schema SHA-256       : a32bbf83a5f954d64c60ee4d737299f86be80692aab773c35ebd6f681f3aa533
API-compatible schema SHA-256            : 527ab2c8b4e76974555b4c8cba99188e00b36a83334c4e190203f46b06238ef0
Cell 7C5 terminal PASS                  : VERIFIED
Cell 7C6 authorization                  : VERIFIED
Cell 7C4 score-blind prompts            : VERIFIED
Cell 7B4 exact LLM/runtime freeze       : VERIFIED
Answer-key access                       : NOT AUTHORIZED
RAG evaluation                          : NOT AUTHORIZED


## 5. Load only authorized score-blind prompts and the exact 1,440-request plan

In [6]:
plan = pd.read_parquet(CELL_7C5['generation_plan']['path'])
prompts = pd.read_parquet(CELL_7C4_PROMPTS['path'])

if len(plan) != EXPECTED_REQUESTS:
    raise AssertionError(f'Generation plan row count changed: {len(plan)}')
if len(prompts) != EXPECTED_PROMPTS:
    raise AssertionError(f'Prompt row count changed: {len(prompts)}')

required_plan_columns = {
    'generation_request_id',
    'prompt_instance_id',
    'question_id',
    'blinded_alias',
    'context_count',
    'system_prompt_sha256',
    'user_prompt_sha256',
    'full_prompt_sha256',
    'run_id',
    'model_snapshot',
    'api',
    'temperature',
    'top_p',
    'max_output_tokens',
    'presence_penalty',
    'frequency_penalty',
    'strict_json_schema',
    'response_schema_name',
    'response_schema_sha256',
    'tools_enabled',
    'web_search_enabled',
    'file_search_enabled',
    'code_interpreter_enabled',
    'answer_key_access_authorized',
    'evaluation_authorized',
}
missing_plan_columns = sorted(required_plan_columns - set(plan.columns))
if missing_plan_columns:
    raise AssertionError(
        'Generation plan is missing required columns: '
        + ', '.join(missing_plan_columns)
    )

required_prompt_columns = {
    'prompt_instance_id',
    'question_id',
    'blinded_alias',
    'context_count',
    'system_prompt',
    'system_prompt_sha256',
    'user_prompt',
    'user_prompt_sha256',
    'full_prompt_sha256',
}
missing_prompt_columns = sorted(required_prompt_columns - set(prompts.columns))
if missing_prompt_columns:
    raise AssertionError(
        'Prompt package is missing required columns: '
        + ', '.join(missing_prompt_columns)
    )

if plan['generation_request_id'].duplicated().any():
    raise AssertionError('Generation request IDs are not unique.')
if prompts['prompt_instance_id'].duplicated().any():
    raise AssertionError('Prompt instance IDs are not unique.')

# Exact plan-level controls.
plan_checks = OrderedDict([
    ('1440_rows', len(plan) == 1440),
    ('480_unique_prompts', plan['prompt_instance_id'].nunique() == 480),
    ('80_questions', plan['question_id'].nunique() == 80),
    ('6_aliases', plan['blinded_alias'].nunique() == 6),
    ('run_ids_exact', set(plan['run_id'].astype(int)) == {0, 1, 2}),
    ('each_prompt_three_requests',
     plan.groupby('prompt_instance_id').size().eq(3).all()),
    ('model_exact', plan['model_snapshot'].eq(FROZEN_MODEL).all()),
    ('api_exact', plan['api'].eq(FROZEN_API).all()),
    ('temperature_exact', plan['temperature'].eq(FROZEN_TEMPERATURE).all()),
    ('top_p_exact', plan['top_p'].eq(FROZEN_TOP_P).all()),
    ('max_tokens_exact',
     plan['max_output_tokens'].eq(FROZEN_MAX_OUTPUT_TOKENS).all()),
    ('presence_penalty_exact',
     plan['presence_penalty'].eq(FROZEN_PRESENCE_PENALTY).all()),
    ('frequency_penalty_exact',
     plan['frequency_penalty'].eq(FROZEN_FREQUENCY_PENALTY).all()),
    ('strict_schema_all_true', plan['strict_json_schema'].eq(True).all()),
    ('tools_all_false', plan['tools_enabled'].eq(False).all()),
    ('web_search_all_false', plan['web_search_enabled'].eq(False).all()),
    ('file_search_all_false', plan['file_search_enabled'].eq(False).all()),
    ('code_interpreter_all_false',
     plan['code_interpreter_enabled'].eq(False).all()),
    ('answer_key_all_false',
     plan['answer_key_access_authorized'].eq(False).all()),
    ('evaluation_all_false',
     plan['evaluation_authorized'].eq(False).all()),
    ('scientific_response_schema_hash_exact',
     plan['response_schema_sha256'].astype(str).eq(
         EXPECTED_SCIENTIFIC_SCHEMA_CANONICAL_SHA256
     ).all()),
])

failed_plan_checks = [
    name for name, passed in plan_checks.items()
    if not bool(passed)
]
if failed_plan_checks:
    raise RuntimeError(
        'Frozen 1,440-request plan validation failed:\\n- '
        + '\\n- '.join(failed_plan_checks)
    )

# Join only score-blind prompt text into the authorized request plan.
prompt_text = prompts[
    [
        'prompt_instance_id',
        'question_id',
        'blinded_alias',
        'system_prompt',
        'system_prompt_sha256',
        'user_prompt',
        'user_prompt_sha256',
        'full_prompt_sha256',
    ]
].copy()

joined = plan.merge(
    prompt_text,
    on=[
        'prompt_instance_id',
        'question_id',
        'blinded_alias',
        'system_prompt_sha256',
        'user_prompt_sha256',
        'full_prompt_sha256',
    ],
    how='left',
    validate='many_to_one',
)

if len(joined) != EXPECTED_REQUESTS:
    raise AssertionError('Plan-to-prompt join changed row count.')
if joined['system_prompt'].isna().any() or joined['user_prompt'].isna().any():
    raise AssertionError('At least one generation request failed to join its frozen prompt.')

# Recompute exact text hashes before any API call.
for _, row in joined.iterrows():
    if sha256_text(str(row['system_prompt'])) != str(row['system_prompt_sha256']):
        raise AssertionError(f'System prompt hash mismatch: {row["generation_request_id"]}')
    if sha256_text(str(row['user_prompt'])) != str(row['user_prompt_sha256']):
        raise AssertionError(f'User prompt hash mismatch: {row["generation_request_id"]}')

# Keep exact frozen plan order.
joined = joined.reset_index(drop=True)

print(f'Frozen score-blind prompts loaded        : {len(prompts):,}')
print(f'Frozen generation requests loaded        : {len(joined):,}')
print('Prompt hashes reproduced                 : YES')
print('Generation-plan scientific schema hash   : VERIFIED')
print('API serialization schema remediation     : VERIFIED')
print('Score-bearing reranking audit loaded     : NO')
print('Cell 7A3 scores loaded                   : NO')
print('Answer keys loaded                       : NO')

Frozen score-blind prompts loaded        : 480
Frozen generation requests loaded        : 1,440
Prompt hashes reproduced                 : YES
Generation-plan scientific schema hash   : VERIFIED
API serialization schema remediation     : VERIFIED
Score-bearing reranking audit loaded     : NO
Cell 7A3 scores loaded                   : NO
Answer keys loaded                       : NO


## 6. API compatibility preflight and secret handling

The frozen Cell 7B4 configuration records `presence_penalty=0.0` and `frequency_penalty=0.0`. Those are neutral values. Cell 7C6 verifies them exactly in the frozen plan.

The Responses API payload is built only from parameters exposed by the pinned SDK. The notebook **never substitutes a nonzero value**. If the pinned Responses API exposes either penalty parameter, the notebook sends the frozen zero explicitly; otherwise the neutral setting is recorded as an API-surface omission rather than inventing an unsupported request field.

No model probe/test call is made. The first network model call is request 1 of the frozen 1,440-request plan.

In [7]:
# Get API key without printing or writing it.
api_key = os.environ.get('OPENAI_API_KEY', '').strip()
if not api_key:
    api_key = getpass('Enter OPENAI_API_KEY (input hidden; never stored by this notebook): ').strip()

if not api_key:
    raise RuntimeError('OPENAI_API_KEY was not provided.')

client = OpenAI(
    api_key=api_key,
    timeout=float(FROZEN_TIMEOUT_SECONDS),
    max_retries=0,  # frozen retry logic is implemented explicitly below
)

create_signature = inspect.signature(client.responses.create)
supported_parameters = set(create_signature.parameters)

required_api_parameters = {
    'model',
    'input',
    'text',
    'temperature',
    'top_p',
    'max_output_tokens',
    'tools',
    'store',
    'stream',
}
missing_required_api_parameters = sorted(
    required_api_parameters - supported_parameters
)
if missing_required_api_parameters:
    raise RuntimeError(
        'Pinned SDK Responses API is missing frozen required request parameters: '
        + ', '.join(missing_required_api_parameters)
    )

PRESENCE_PENALTY_SUPPORTED = 'presence_penalty' in supported_parameters
FREQUENCY_PENALTY_SUPPORTED = 'frequency_penalty' in supported_parameters

# Since both frozen values are exactly neutral zero, absence from the Responses API
# surface does not authorize a substitute value.
if FROZEN_PRESENCE_PENALTY != 0.0 or FROZEN_FREQUENCY_PENALTY != 0.0:
    raise RuntimeError(
        'Penalty-parameter compatibility policy only permits neutral frozen values.'
    )

print('Pinned OpenAI SDK Responses.create preflight : PASS')
print(f'presence_penalty exposed by SDK               : {PRESENCE_PENALTY_SUPPORTED}')
print(f'frequency_penalty exposed by SDK              : {FREQUENCY_PENALTY_SUPPORTED}')
print('Frozen penalty values                          : 0.0 / 0.0')
print('API key                                         : LOADED BUT NOT DISPLAYED OR STORED')
print('Preflight model calls                           : 0')

Enter OPENAI_API_KEY (input hidden; never stored by this notebook): ··········
Pinned OpenAI SDK Responses.create preflight : PASS
presence_penalty exposed by SDK               : False
frequency_penalty exposed by SDK              : False
Frozen penalty values                          : 0.0 / 0.0
API key                                         : LOADED BUT NOT DISPLAYED OR STORED
Preflight model calls                           : 0


## 7. Strict structured-output observation and schema-validation helpers

In [8]:
def validate_structured_output(payload: Any) -> tuple[bool, list[str]]:
    # Scientific validator follows the ORIGINAL Cell 7B4 contract, not the API projection.
    errors: list[str] = []

    if not isinstance(payload, dict):
        return False, ['parsed output is not a JSON object']

    observed_keys = set(payload.keys())
    if observed_keys != EXPECTED_RESPONSE_FIELDS:
        errors.append(
            'field set mismatch: '
            f'expected={sorted(EXPECTED_RESPONSE_FIELDS)}, '
            f'observed={sorted(observed_keys)}'
        )

    if not isinstance(payload.get('answer'), str):
        errors.append('answer must be string')

    if not isinstance(payload.get('clinical_significance'), str):
        errors.append('clinical_significance must be string')

    if not isinstance(payload.get('conflict_detected'), bool):
        errors.append('conflict_detected must be boolean')

    if payload.get('evidence_strength') not in {'high', 'moderate', 'low'}:
        errors.append('evidence_strength enum invalid')

    if payload.get('response_policy') not in {'answer', 'cautious_answer', 'abstain'}:
        errors.append('response_policy enum invalid')

    confidence = payload.get('confidence')
    if (
        isinstance(confidence, bool)
        or not isinstance(confidence, (int, float))
        or not np.isfinite(float(confidence))
        or not (0.0 <= float(confidence) <= 1.0)
    ):
        errors.append('confidence must be finite number in [0,1]')

    evidence_ids = payload.get('evidence_ids')
    if not isinstance(evidence_ids, list):
        errors.append('evidence_ids must be array')
    else:
        if not all(isinstance(value, str) for value in evidence_ids):
            errors.append('evidence_ids entries must all be strings')
        if len(evidence_ids) != len(set(evidence_ids)):
            errors.append('evidence_ids must contain unique values')  # Cell 7C5R mandatory post-response validator

    if not isinstance(payload.get('reasoning_summary'), str):
        errors.append('reasoning_summary must be string')

    return len(errors) == 0, errors


def extract_response_observation(response: Any) -> dict[str, Any]:
    raw = to_json_native(response.model_dump())
    status = str(getattr(response, 'status', '') or '')

    incomplete_details = getattr(response, 'incomplete_details', None)
    incomplete_reason = None
    if incomplete_details is not None:
        if hasattr(incomplete_details, 'reason'):
            incomplete_reason = getattr(incomplete_details, 'reason')
        elif isinstance(incomplete_details, dict):
            incomplete_reason = incomplete_details.get('reason')
        if incomplete_reason is not None:
            incomplete_reason = str(incomplete_reason)

    output_text = str(getattr(response, 'output_text', '') or '')

    refusal_texts: list[str] = []
    try:
        for item in getattr(response, 'output', []) or []:
            if getattr(item, 'type', None) != 'message':
                continue
            for content in getattr(item, 'content', []) or []:
                if getattr(content, 'type', None) == 'refusal':
                    refusal_value = getattr(content, 'refusal', None)
                    if refusal_value:
                        refusal_texts.append(str(refusal_value))
    except Exception:
        # Raw response is still preserved even if convenience extraction changes.
        pass

    parsed = None
    parse_error = None
    structured_valid = False
    schema_errors: list[str] = []

    if output_text:
        try:
            parsed = json.loads(output_text)
        except Exception as exc:
            parse_error = f'{type(exc).__name__}: {exc}'
        else:
            structured_valid, schema_errors = validate_structured_output(parsed)

    usage = getattr(response, 'usage', None)
    usage_dict = to_json_native(usage.model_dump()) if hasattr(usage, 'model_dump') else {}

    return {
        'response_id': str(getattr(response, 'id', '') or ''),
        'response_status': status,
        'incomplete_reason': incomplete_reason,
        'refusal_detected': bool(refusal_texts),
        'refusal_texts': refusal_texts,
        'output_text': output_text,
        'output_text_sha256': sha256_text(output_text) if output_text else None,
        'parsed_output': parsed,
        'json_parse_error': parse_error,
        'structured_valid': bool(structured_valid),
        'schema_errors': schema_errors,
        'usage': usage_dict,
        'raw_response': raw,
        'raw_response_sha256': sha256_text(canonical_json_text(raw)),
    }


def usage_value(usage: dict[str, Any], key: str) -> int | None:
    value = usage.get(key)
    if value is None:
        return None
    try:
        return int(value)
    except Exception:
        return None

## 8. Exact frozen request execution with transport/rate-limit-only retries and restart-safe checkpoints

In [9]:
def build_request_kwargs(row: pd.Series) -> dict[str, Any]:
    request_kwargs: dict[str, Any] = {
        'model': FROZEN_MODEL,
        'input': [
            {
                'role': 'system',
                'content': str(row['system_prompt']),
            },
            {
                'role': 'user',
                'content': str(row['user_prompt']),
            },
        ],
        'text': {
            'format': {
                'type': 'json_schema',
                'name': str(structured_cfg['name']),
                'schema': api_response_schema,
                'strict': True,
            },
        },
        'temperature': FROZEN_TEMPERATURE,
        'top_p': FROZEN_TOP_P,
        'max_output_tokens': FROZEN_MAX_OUTPUT_TOKENS,
        'tools': [],
        'store': False,
        'stream': False,
    }

    # Explicitly pass only when the pinned Responses API exposes these parameters.
    if PRESENCE_PENALTY_SUPPORTED:
        request_kwargs['presence_penalty'] = FROZEN_PRESENCE_PENALTY
    if FREQUENCY_PENALTY_SUPPORTED:
        request_kwargs['frequency_penalty'] = FROZEN_FREQUENCY_PENALTY

    return request_kwargs


def load_valid_checkpoint(row: pd.Series) -> dict[str, Any] | None:
    path = checkpoint_path_for_request(str(row['generation_request_id']))
    if not path.exists():
        return None

    if not sidecar_is_valid(path):
        raise RuntimeError(
            f'Existing checkpoint has missing/invalid sidecar; fail closed: {path}'
        )

    checkpoint = load_json(path)
    expected_fingerprint = request_fingerprint(row)
    expected_idempotency = idempotency_key_for_request(row)

    checks = {
        'generation_request_id':
            checkpoint.get('generation_request_id') == str(row['generation_request_id']),
        'request_fingerprint':
            checkpoint.get('request_fingerprint') == expected_fingerprint,
        'idempotency_key':
            checkpoint.get('idempotency_key') == expected_idempotency,
        'model_snapshot':
            checkpoint.get('model_snapshot') == FROZEN_MODEL,
        'full_prompt_sha256':
            checkpoint.get('full_prompt_sha256') == str(row['full_prompt_sha256']),
        'api_response_received':
            checkpoint.get('api_response_received') is True,
    }

    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        raise RuntimeError(
            f'Checkpoint provenance mismatch for {row["generation_request_id"]}: '
            + ', '.join(failed)
        )

    return checkpoint


def save_checkpoint(
    row: pd.Series,
    observation: dict[str, Any],
    transport_retries_used: int,
    request_started_utc: str,
    response_received_utc: str,
) -> dict[str, Any]:
    checkpoint = {
        'checkpoint_version': '2.0.0',
        'cell_id': CELL_ID,
        'generation_request_id': str(row['generation_request_id']),
        'prompt_instance_id': str(row['prompt_instance_id']),
        'question_id': str(row['question_id']),
        'blinded_alias': str(row['blinded_alias']),
        'run_id': int(row['run_id']),
        'model_snapshot': FROZEN_MODEL,
        'api': FROZEN_API,
        'full_prompt_sha256': str(row['full_prompt_sha256']),
        'request_fingerprint': request_fingerprint(row),
        'idempotency_key': idempotency_key_for_request(row),
        'request_started_utc': request_started_utc,
        'response_received_utc': response_received_utc,
        'transport_retries_used': int(transport_retries_used),
        'api_response_received': True,
        'observation': observation,
    }

    path = checkpoint_path_for_request(str(row['generation_request_id']))
    stable_write_json(path, checkpoint)
    write_sidecar(path)

    if not sidecar_is_valid(path):
        raise AssertionError(f'Checkpoint readback failed: {path}')

    readback = load_json(path)
    if readback.get('request_fingerprint') != checkpoint['request_fingerprint']:
        raise AssertionError(f'Checkpoint fingerprint readback failed: {path}')

    return checkpoint


all_checkpoints: list[dict[str, Any]] = []
new_api_calls_this_run = 0
reused_checkpoints_this_run = 0
total_transport_retries_this_run = 0

progress = tqdm(
    total=EXPECTED_REQUESTS,
    desc='Cell 7C6 frozen generation requests',
    unit='request',
)

for _, row in joined.iterrows():
    existing = load_valid_checkpoint(row)
    if existing is not None:
        all_checkpoints.append(existing)
        reused_checkpoints_this_run += 1
        progress.update(1)
        continue

    request_kwargs = build_request_kwargs(row)
    idempotency_key = idempotency_key_for_request(row)

    response = None
    retries_used = 0
    request_started_utc = datetime.now(timezone.utc).isoformat()

    for attempt in range(FROZEN_MAX_TRANSPORT_RETRIES + 1):
        try:
            response = client.responses.create(
                **request_kwargs,
                extra_headers={
                    'Idempotency-Key': idempotency_key,
                },
            )
            break
        except (RateLimitError, APIConnectionError, APITimeoutError) as exc:
            if attempt >= FROZEN_MAX_TRANSPORT_RETRIES:
                raise RuntimeError(
                    'Frozen transport/rate-limit retry budget exhausted for '
                    f'{row["generation_request_id"]}. '
                    'No final Cell 7C6 package was written; completed checkpoints remain reusable.'
                ) from exc

            sleep_seconds = FROZEN_RETRY_BACKOFF_SECONDS[attempt]
            retries_used += 1
            total_transport_retries_this_run += 1
            time.sleep(sleep_seconds)

    if response is None:
        raise RuntimeError(
            f'No API response object obtained for {row["generation_request_id"]}.'
        )

    new_api_calls_this_run += 1
    response_received_utc = datetime.now(timezone.utc).isoformat()
    observation = extract_response_observation(response)

    checkpoint = save_checkpoint(
        row=row,
        observation=observation,
        transport_retries_used=retries_used,
        request_started_utc=request_started_utc,
        response_received_utc=response_received_utc,
    )
    all_checkpoints.append(checkpoint)

    # Deliberately do not print prompt or response contents.
    progress.update(1)

progress.close()

if len(all_checkpoints) != EXPECTED_REQUESTS:
    raise AssertionError(
        f'Checkpoint accounting mismatch: {len(all_checkpoints)} != {EXPECTED_REQUESTS}'
    )

# Release secret from notebook variable as soon as generation completes.
api_key = None

print(f'Frozen requests accounted for             : {len(all_checkpoints):,}/{EXPECTED_REQUESTS:,}')
print(f'New API calls this run                    : {new_api_calls_this_run:,}')
print(f'Valid checkpoints reused this run         : {reused_checkpoints_this_run:,}')
print(f'Transport/rate-limit retries this run     : {total_transport_retries_this_run:,}')
print('Prompt contents displayed                : NO')
print('Response contents displayed              : NO')
print('Answer keys loaded                       : NO')
print('RAG metrics calculated                   : NO')

Cell 7C6 frozen generation requests:   0%|          | 0/1440 [00:00<?, ?request/s]

Frozen requests accounted for             : 1,440/1,440
New API calls this run                    : 1,440
Valid checkpoints reused this run         : 0
Transport/rate-limit retries this run     : 0
Prompt contents displayed                : NO
Response contents displayed              : NO
Answer keys loaded                       : NO
RAG metrics calculated                   : NO


## 9. Build raw-response, operational-inventory, and structured-output artifacts without evaluation

In [10]:
# Index checkpoints by immutable request ID and materialize in exact frozen plan order.
checkpoint_by_id = {
    str(cp['generation_request_id']): cp
    for cp in all_checkpoints
}
if len(checkpoint_by_id) != EXPECTED_REQUESTS:
    raise AssertionError('Checkpoint generation_request_id values are not unique.')

raw_records: list[dict[str, Any]] = []
inventory_rows: list[dict[str, Any]] = []
structured_rows: list[dict[str, Any]] = []

for _, row in joined.iterrows():
    request_id = str(row['generation_request_id'])
    cp = checkpoint_by_id[request_id]
    obs = cp['observation']

    raw_records.append({
        'generation_request_id': request_id,
        'prompt_instance_id': str(row['prompt_instance_id']),
        'question_id': str(row['question_id']),
        'blinded_alias': str(row['blinded_alias']),
        'run_id': int(row['run_id']),
        'model_snapshot': FROZEN_MODEL,
        'full_prompt_sha256': str(row['full_prompt_sha256']),
        'request_fingerprint': str(cp['request_fingerprint']),
        'idempotency_key': str(cp['idempotency_key']),
        'response_id': obs.get('response_id'),
        'response_status': obs.get('response_status'),
        'raw_response': obs.get('raw_response'),
    })

    usage = obs.get('usage') or {}
    inventory_rows.append({
        'generation_request_id': request_id,
        'prompt_instance_id': str(row['prompt_instance_id']),
        'question_id': str(row['question_id']),
        'blinded_alias': str(row['blinded_alias']),
        'run_id': int(row['run_id']),
        'model_snapshot': FROZEN_MODEL,
        'full_prompt_sha256': str(row['full_prompt_sha256']),
        'request_fingerprint': str(cp['request_fingerprint']),
        'idempotency_key': str(cp['idempotency_key']),
        'response_id': obs.get('response_id'),
        'response_status': obs.get('response_status'),
        'incomplete_reason': obs.get('incomplete_reason'),
        'refusal_detected': bool(obs.get('refusal_detected', False)),
        'output_text_sha256': obs.get('output_text_sha256'),
        'raw_response_sha256': obs.get('raw_response_sha256'),
        'json_parse_error_present': obs.get('json_parse_error') is not None,
        'structured_valid': bool(obs.get('structured_valid', False)),
        'schema_error_count': int(len(obs.get('schema_errors') or [])),
        'transport_retries_used': int(cp.get('transport_retries_used', 0)),
        'request_started_utc': cp.get('request_started_utc'),
        'response_received_utc': cp.get('response_received_utc'),
        'input_tokens': usage_value(usage, 'input_tokens'),
        'output_tokens': usage_value(usage, 'output_tokens'),
        'total_tokens': usage_value(usage, 'total_tokens'),
    })

    parsed = obs.get('parsed_output')
    valid = bool(obs.get('structured_valid', False))
    structured_rows.append({
        'generation_request_id': request_id,
        'prompt_instance_id': str(row['prompt_instance_id']),
        'question_id': str(row['question_id']),
        'blinded_alias': str(row['blinded_alias']),
        'run_id': int(row['run_id']),
        'full_prompt_sha256': str(row['full_prompt_sha256']),
        'response_id': obs.get('response_id'),
        'response_status': obs.get('response_status'),
        'incomplete_reason': obs.get('incomplete_reason'),
        'refusal_detected': bool(obs.get('refusal_detected', False)),
        'structured_valid': valid,
        'json_parse_error': obs.get('json_parse_error'),
        'schema_errors_json': canonical_json_text(obs.get('schema_errors') or []),
        'answer': parsed.get('answer') if valid else None,
        'clinical_significance': parsed.get('clinical_significance') if valid else None,
        'conflict_detected': parsed.get('conflict_detected') if valid else None,
        'evidence_strength': parsed.get('evidence_strength') if valid else None,
        'response_policy': parsed.get('response_policy') if valid else None,
        'confidence': float(parsed.get('confidence')) if valid else None,
        'evidence_ids': parsed.get('evidence_ids') if valid else None,
        'reasoning_summary': parsed.get('reasoning_summary') if valid else None,
        'output_text_sha256': obs.get('output_text_sha256'),
    })

response_inventory = pd.DataFrame(inventory_rows)
structured_outputs = pd.DataFrame(structured_rows)

if len(response_inventory) != EXPECTED_REQUESTS:
    raise AssertionError('Response inventory does not contain exactly 1,440 rows.')
if len(structured_outputs) != EXPECTED_REQUESTS:
    raise AssertionError('Structured-output table does not contain exactly 1,440 rows.')
if response_inventory['generation_request_id'].duplicated().any():
    raise AssertionError('Response inventory contains duplicate request IDs.')
if structured_outputs['generation_request_id'].duplicated().any():
    raise AssertionError('Structured-output table contains duplicate request IDs.')

structured_valid_count = int(response_inventory['structured_valid'].sum())
refusal_count = int(response_inventory['refusal_detected'].sum())
incomplete_count = int(
    response_inventory['response_status'].astype(str).eq('incomplete').sum()
)
json_parse_error_count = int(response_inventory['json_parse_error_present'].sum())

# Operational token accounting only — not scientific RAG performance.
input_token_total = int(
    pd.to_numeric(response_inventory['input_tokens'], errors='coerce')
    .fillna(0).sum()
)
output_token_total = int(
    pd.to_numeric(response_inventory['output_tokens'], errors='coerce')
    .fillna(0).sum()
)
total_token_total = int(
    pd.to_numeric(response_inventory['total_tokens'], errors='coerce')
    .fillna(0).sum()
)

print(f'API responses preserved                  : {len(response_inventory):,}')
print(f'Structured-valid responses               : {structured_valid_count:,}')
print(f'Refusals preserved                       : {refusal_count:,}')
print(f'Incomplete responses preserved           : {incomplete_count:,}')
print(f'JSON parse-error observations preserved  : {json_parse_error_count:,}')
print('Answer correctness evaluated             : NO')
print('Citation correctness evaluated           : NO')
print('Condition identity unblinded             : NO')

API responses preserved                  : 1,440
Structured-valid responses               : 1,440
Refusals preserved                       : 0
Incomplete responses preserved           : 0
JSON parse-error observations preserved  : 0
Answer correctness evaluated             : NO
Citation correctness evaluated           : NO
Condition identity unblinded             : NO


## 10. Final execution QC, frozen package write, immutable readback, and terminal boundary

In [11]:
created_utc = datetime.now(timezone.utc).isoformat()

# Final execution QC concerns execution integrity only. It deliberately does not require
# every model response to be a valid structured answer; refusals/incomplete outputs remain observations.
execution_checks = OrderedDict([
    ('cell_7c5_five_artifacts_verified',
     sum(1 for x in verified_inputs if x.get('source_cell') == '7C5') == 5),
    ('cell_7c5_terminal_pass_exact',
     manifest_7c5.get('terminal_decision') == EXPECTED_CELL_7C5_TERMINAL_DECISION),
    ('cell_7c5r_terminal_pass_exact',
     manifest_7c5r.get('terminal_decision') == EXPECTED_CELL_7C5R_TERMINAL_DECISION),
    ('cell_7c6_v2_remediation_authorization_exact',
     authorization_7c5r.get('authorization_decision') == EXPECTED_CELL_7C5R_AUTHORIZATION_DECISION),
    ('api_schema_canonical_hash_exact',
     api_schema_sha256 == EXPECTED_API_SCHEMA_CANONICAL_SHA256),
    ('scientific_schema_canonical_hash_exact',
     scientific_schema_sha256 == EXPECTED_SCIENTIFIC_SCHEMA_CANONICAL_SHA256),
    ('api_projection_only_removes_uniqueItems',
     reconstructed_scientific_schema == scientific_response_schema),
    ('plan_rows_1440', len(joined) == 1440),
    ('checkpoint_rows_1440', len(all_checkpoints) == 1440),
    ('response_inventory_rows_1440', len(response_inventory) == 1440),
    ('structured_output_rows_1440', len(structured_outputs) == 1440),
    ('unique_generation_request_ids',
     response_inventory['generation_request_id'].nunique() == 1440),
    ('every_plan_request_observed',
     set(joined['generation_request_id'].astype(str)) ==
     set(response_inventory['generation_request_id'].astype(str))),
    ('frozen_model_only',
     response_inventory['model_snapshot'].eq(FROZEN_MODEL).all()),
    ('prompt_hashes_nonmissing',
     response_inventory['full_prompt_sha256'].notna().all()),
    ('raw_hashes_nonmissing',
     response_inventory['raw_response_sha256'].notna().all()),
    ('response_ids_nonmissing',
     response_inventory['response_id'].astype(str).str.len().gt(0).all()),
    ('answer_keys_not_loaded', True),
    ('score_bearing_audit_not_loaded', True),
    ('cell7a3_scores_not_loaded', True),
    ('adjudication_not_performed', True),
    ('rag_metrics_not_calculated', True),
])

failed_execution_checks = [
    name for name, passed in execution_checks.items()
    if not bool(passed)
]
if failed_execution_checks:
    raise RuntimeError(
        'Cell 7C6 execution-integrity QC failed:\\n- '
        + '\\n- '.join(failed_execution_checks)
    )

input_inventory = pd.DataFrame(verified_inputs)

execution_report = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': created_utc,
    'notebook': NOTEBOOK_NAME,
    'project_root': str(ROOT),
    'authorization': {
        'cell_7c5_manifest_sha256': CELL_7C5['manifest']['sha256'],
        'cell_7c5_authorization_sha256': CELL_7C5['authorization']['sha256'],
        'cell_7c5r_manifest_sha256': CELL_7C5R['manifest']['sha256'],
        'cell_7c5r_remediation_authorization_sha256': CELL_7C5R['remediation_authorization']['sha256'],
        'authorization_decision': EXPECTED_CELL_7C5R_AUTHORIZATION_DECISION,
        'authorized_cell': '7C6_V2',
    },
    'frozen_generation_design': {
        'prompt_instances': EXPECTED_PROMPTS,
        'run_ids': EXPECTED_RUN_IDS,
        'planned_requests': EXPECTED_REQUESTS,
        'model_snapshot': FROZEN_MODEL,
        'api': FROZEN_API,
        'temperature': FROZEN_TEMPERATURE,
        'top_p': FROZEN_TOP_P,
        'max_output_tokens': FROZEN_MAX_OUTPUT_TOKENS,
        'presence_penalty': FROZEN_PRESENCE_PENALTY,
        'frequency_penalty': FROZEN_FREQUENCY_PENALTY,
        'presence_penalty_exposed_by_pinned_sdk': PRESENCE_PENALTY_SUPPORTED,
        'frequency_penalty_exposed_by_pinned_sdk': FREQUENCY_PENALTY_SUPPORTED,
        'strict_json_schema': True,
        'scientific_response_schema_sha256_canonical': scientific_schema_sha256,
        'api_response_schema_sha256_canonical': api_schema_sha256,
        'api_schema_remediation': 'removed only properties.evidence_ids.uniqueItems',
        'post_response_evidence_ids_uniqueness_validation_required': True,
        'tools': [],
        'store': False,
        'stream': False,
        'timeout_seconds': FROZEN_TIMEOUT_SECONDS,
        'maximum_transport_retries': FROZEN_MAX_TRANSPORT_RETRIES,
        'retry_backoff_seconds': FROZEN_RETRY_BACKOFF_SECONDS,
    },
    'execution_accounting': {
        'planned_requests': EXPECTED_REQUESTS,
        'completed_api_response_observations': int(len(response_inventory)),
        'new_api_calls_this_run': int(new_api_calls_this_run),
        'reused_valid_checkpoints_this_run': int(reused_checkpoints_this_run),
        'transport_rate_limit_retries_this_run': int(total_transport_retries_this_run),
        'structured_valid_responses': structured_valid_count,
        'refusal_responses': refusal_count,
        'incomplete_responses': incomplete_count,
        'json_parse_error_responses': json_parse_error_count,
    },
    'operational_token_accounting': {
        'input_tokens_total': input_token_total,
        'output_tokens_total': output_token_total,
        'total_tokens_total': total_token_total,
        'scientific_performance_metric': False,
    },
    'scientific_operations': {
        'llm_called': True,
        'responses_generated': True,
        'score_bearing_cell7c2_audit_loaded': False,
        'cell7a3_scores_loaded': False,
        'answer_key_outcomes_inspected': False,
        'condition_identity_unblinded': False,
        'adjudication_performed': False,
        'rag_metrics_calculated': False,
        'bootstrap_or_inference_performed': False,
    },
    'checkpoint_policy': {
        'directory': str(CHECKPOINT_DIR),
        'recovery_only': True,
        'excluded_from_final_manifest_artifact_set': True,
        'valid_checkpoint_requires_exact_request_fingerprint_and_sha256_sidecar': True,
    },
}

terminal_decision = (
    'PASS_STAGE7C6_V2_1440_FROZEN_SCORE_BLIND_LLM_GENERATION_REQUESTS_COMPLETED_'
    'USING_CELL7C5R_API_COMPATIBLE_STRUCTURED_OUTPUT_SCHEMA_WITH_ORIGINAL_SCIENTIFIC_'
    'EVIDENCE_ID_UNIQUENESS_VALIDATION_PRESERVED_RAW_AND_STRUCTURED_RESPONSES_'
    'MATERIALIZED_CHECKSUM_PROTECTED_CHECKPOINT_RESUMABLE_NO_SCORE_BEARING_AUDIT_'
    'CELL7A3_SCORES_ANSWER_KEYS_ADJUDICATION_OR_RAG_METRICS_NEXT_EXECUTION_NOT_AUTHORIZED'
)

qc_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': created_utc,
    'frozen_config_checks': {
        name: bool(value) for name, value in frozen_config_checks.items()
    },
    'plan_checks': {
        name: bool(value) for name, value in plan_checks.items()
    },
    'execution_checks': {
        name: bool(value) for name, value in execution_checks.items()
    },
    'response_observation_counts': {
        'structured_valid': structured_valid_count,
        'refusal': refusal_count,
        'incomplete': incomplete_count,
        'json_parse_error': json_parse_error_count,
    },
    'important_note': (
        'Response-observation counts are not pass/fail scientific metrics. '
        'Refusals, incomplete responses, and schema-invalid outputs are retained without preferred-answer retries.'
    ),
    'failed_checks': 0,
    'terminal_decision': terminal_decision,
}

# Stage all final artifacts in a temporary directory first.
with tempfile.TemporaryDirectory(prefix='cell_7c6_final_staging_') as staging_text:
    staging = Path(staging_text)

    staged = OrderedDict([
        ('raw_response_envelopes', staging / OUTPUTS['raw_response_envelopes'].name),
        ('response_inventory', staging / OUTPUTS['response_inventory'].name),
        ('structured_outputs', staging / OUTPUTS['structured_outputs'].name),
        ('input_inventory', staging / OUTPUTS['input_inventory'].name),
        ('execution_report', staging / OUTPUTS['execution_report'].name),
        ('qc', staging / OUTPUTS['qc'].name),
    ])

    stable_write_jsonl(staged['raw_response_envelopes'], raw_records)
    stable_write_parquet(staged['response_inventory'], response_inventory)
    stable_write_parquet(staged['structured_outputs'], structured_outputs)
    stable_write_csv(staged['input_inventory'], input_inventory)
    stable_write_json(staged['execution_report'], execution_report)
    stable_write_json(staged['qc'], qc_payload)

    staged_hashes = {
        key: sha256_file(path)
        for key, path in staged.items()
    }

    manifest_payload = {
        'cell_id': CELL_ID,
        'stage': STAGE,
        'package_version': PACKAGE_VERSION,
        'created_utc': created_utc,
        'notebook': NOTEBOOK_NAME,
        'project_root': str(ROOT),
        'authorization_lineage': {
            'cell_7c5_authorization_sha256': CELL_7C5['authorization']['sha256'],
            'cell_7c5_generation_plan_sha256': CELL_7C5['generation_plan']['sha256'],
            'cell_7c5_manifest_sha256': CELL_7C5['manifest']['sha256'],
            'cell_7c5r_manifest_sha256': CELL_7C5R['manifest']['sha256'],
            'cell_7c5r_remediation_authorization_sha256': CELL_7C5R['remediation_authorization']['sha256'],
            'cell_7c5r_api_compatible_schema_file_sha256': CELL_7C5R['api_compatible_schema']['sha256'],
            'scientific_response_schema_sha256_canonical': scientific_schema_sha256,
            'api_response_schema_sha256_canonical': api_schema_sha256,
            'cell_7c4_prompt_instances_sha256': CELL_7C4_PROMPTS['sha256'],
            'cell_7b4_llm_prompt_response_sha256': CELL_7B4_LLM['sha256'],
            'cell_7b4_runtime_determinism_sha256': CELL_7B4_RUNTIME['sha256'],
        },
        'generation_package': {
            'planned_requests': EXPECTED_REQUESTS,
            'completed_response_observations': int(len(response_inventory)),
            'model_snapshot': FROZEN_MODEL,
            'structured_valid_responses': structured_valid_count,
            'refusal_responses': refusal_count,
            'incomplete_responses': incomplete_count,
            'json_parse_error_responses': json_parse_error_count,
        },
        'output_artifacts': {
            key: {
                'path': str(OUTPUTS[key]),
                'sha256': staged_hashes[key],
            }
            for key in staged
        },
        'recovery_checkpoints_in_manifest': False,
        'answer_key_access_authorized': False,
        'evaluation_authorized': False,
        'next_authorized_cell': None,
        'next_required_action': (
            'Separate fail-closed authorization before loading answer keys, unblinding condition identities, '
            'adjudication, run aggregation, or any RAG-performance metric.'
        ),
        'terminal_decision': terminal_decision,
    }

    staged_manifest = staging / OUTPUTS['manifest'].name
    stable_write_json(staged_manifest, manifest_payload)

    # Copy staged scientific package to final locations. Manifest is committed last.
    commit_order = [
        'raw_response_envelopes',
        'response_inventory',
        'structured_outputs',
        'input_inventory',
        'execution_report',
        'qc',
    ]
    for key in commit_order:
        shutil.copyfile(staged[key], OUTPUTS[key])
        write_sidecar(OUTPUTS[key])

    shutil.copyfile(staged_manifest, OUTPUTS['manifest'])
    write_sidecar(OUTPUTS['manifest'])

# Fresh readback of all seven frozen artifacts.
for key, path in OUTPUTS.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing final Cell 7C6 artifact: {path}')
    if not sidecar_is_valid(path):
        raise AssertionError(f'Final Cell 7C6 sidecar invalid: {path}')

readback_inventory = pd.read_parquet(OUTPUTS['response_inventory'])
readback_structured = pd.read_parquet(OUTPUTS['structured_outputs'])
readback_manifest = load_json(OUTPUTS['manifest'])
readback_qc = load_json(OUTPUTS['qc'])

readback_checks = OrderedDict([
    ('response_inventory_1440', len(readback_inventory) == 1440),
    ('structured_outputs_1440', len(readback_structured) == 1440),
    ('response_inventory_unique_ids',
     readback_inventory['generation_request_id'].nunique() == 1440),
    ('structured_outputs_unique_ids',
     readback_structured['generation_request_id'].nunique() == 1440),
    ('manifest_terminal_exact',
     readback_manifest.get('terminal_decision') == terminal_decision),
    ('manifest_next_cell_none',
     readback_manifest.get('next_authorized_cell') is None),
    ('manifest_answer_key_false',
     readback_manifest.get('answer_key_access_authorized') is False),
    ('manifest_evaluation_false',
     readback_manifest.get('evaluation_authorized') is False),
    ('qc_zero_failures',
     int(readback_qc.get('failed_checks', -1)) == 0),
    ('all_seven_sidecars_valid',
     all(sidecar_is_valid(path) for path in OUTPUTS.values())),
])

failed_readback = [
    name for name, passed in readback_checks.items()
    if not bool(passed)
]
if failed_readback:
    raise RuntimeError(
        'Cell 7C6 final readback QC failed:\\n- '
        + '\\n- '.join(failed_readback)
    )

total_qc_checks = (
    len(frozen_config_checks)
    + len(plan_checks)
    + len(execution_checks)
    + len(readback_checks)
)

separator = '=' * 158
print('\\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C6 V2')
print('EXACT FROZEN SCORE-BLIND LLM GENERATION EXECUTION — API-COMPATIBLE SCHEMA')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\\nUPSTREAM AUTHORIZATION AND FREEZE REVERIFICATION')
print(f'Cell 7C5 manifest SHA-256                     : {CELL_7C5["manifest"]["sha256"]}')
print('Cell 7C5 terminal PASS verified               : YES')
print('Cell 7C5R remediation manifest SHA-256        : ' + CELL_7C5R['manifest']['sha256'])
print('Cell 7C6 V2 remediation authorization         : VERIFIED')
print(f'Original scientific schema SHA-256            : {scientific_schema_sha256}')
print(f'API-compatible schema SHA-256                 : {api_schema_sha256}')
print('API schema change                             : evidence_ids.uniqueItems removed only')
print('Post-response uniqueness validation           : REQUIRED')
print(f'Cell 7C4 prompt package SHA-256                : {CELL_7C4_PROMPTS["sha256"]}')
print(f'Cell 7B4 LLM config SHA-256                    : {CELL_7B4_LLM["sha256"]}')
print('Score-bearing Cell 7C2 audit loaded           : NO')
print('Cell 7A3 scores loaded                        : NO')
print('Answer keys loaded                            : NO')

print('\\nFROZEN GENERATION EXECUTION')
print(f'Model snapshot                                : {FROZEN_MODEL}')
print(f'API                                           : {FROZEN_API}')
print(f'OpenAI SDK                                    : {FROZEN_OPENAI_VERSION}')
print(f'Temperature / top-p                           : {FROZEN_TEMPERATURE} / {FROZEN_TOP_P}')
print(f'Max output tokens                             : {FROZEN_MAX_OUTPUT_TOKENS}')
print(f'Frozen prompt instances                       : {EXPECTED_PROMPTS}')
print(f'Run IDs                                       : {EXPECTED_RUN_IDS}')
print(f'Planned generation requests                   : {EXPECTED_REQUESTS:,}')
print(f'Completed API response observations           : {len(readback_inventory):,}')
print(f'New API calls this run                        : {new_api_calls_this_run:,}')
print(f'Valid checkpoints reused this run             : {reused_checkpoints_this_run:,}')
print(f'Transport/rate-limit retries this run         : {total_transport_retries_this_run:,}')
print('Prompt contents displayed                     : NO')
print('Response contents displayed                   : NO')

print('\\nRESPONSE OBSERVATION ACCOUNTING — NOT PERFORMANCE EVALUATION')
print(f'Structured-valid responses                    : {structured_valid_count:,}')
print(f'Refusal responses                             : {refusal_count:,}')
print(f'Incomplete responses                          : {incomplete_count:,}')
print(f'JSON parse-error responses                    : {json_parse_error_count:,}')
print(f'Operational input tokens                      : {input_token_total:,}')
print(f'Operational output tokens                     : {output_token_total:,}')
print(f'Operational total tokens                      : {total_token_total:,}')

print('\\nCELL 7C6 FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\\nQC checks                                      : {total_qc_checks}/{total_qc_checks} PASS')

print('\\nSCIENTIFIC OPERATIONS IN CELL 7C6')
print('LLM called                                    : YES')
print('Responses generated                           : YES')
print('Raw responses preserved                       : YES')
print('Structured response outputs preserved         : YES')
print('Answer-key outcomes inspected                 : NO')
print('Condition identities unblinded                : NO')
print('Adjudication                                  : NO')
print('RAG metrics / bootstrap inference             : NO')

print('\\nNEXT AUTHORIZATION BOUNDARY')
print('Next execution cell                           : NOT AUTHORIZED')
print('Required next action                          : separate fail-closed authorization before')
print('                                                 answer-key loading, unblinding, adjudication,')
print('                                                 run aggregation, or RAG-performance evaluation')

print(f'\\nFINAL DECISION                                : {terminal_decision}')
print(separator)

\n==============================================================================================================================================================
EXPERIMENT 2 — STAGE 7C — CELL 7C6 V2
EXACT FROZEN SCORE-BLIND LLM GENERATION EXECUTION — API-COMPATIBLE SCHEMA
Notebook                                      : 13_GES_Aware_Genomic_RAG_Cell_7C6_V2_Exact_LLM_Generation_Execution_API_Compatible.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study
\nUPSTREAM AUTHORIZATION AND FREEZE REVERIFICATION
Cell 7C5 manifest SHA-256                     : 3bfb3a24fff0eb1ace3ff83d994c4f9ea4f0d0492ae014739378311fb48a51df
Cell 7C5 terminal PASS verified               : YES
Cell 7C5R remediation manifest SHA-256        : 580ef084f0e5470b83645fdc6334ff8f385fb93ae42502e98dbac180f2259b63
Cell 7C6 V2 remediation authorization         : VERIFIED
Original scientific schema SHA-256            : a32bbf83a5f954d64c60ee4d737299f86be80692aab773c35ebd6f681f3aa5